In [ ]:
# importing reqd libs
import cv2
import torch
from tracker import DeepSORT_Tracker
from utils import capture_setter, get_person_detections, save_tracklets
from osnet_log_generator import generate_log_from_tracklets

# reid specific dependnecies
import torch
import torchreid
import torchvision.transforms as T
from PIL import Image
import numpy as np

# inputs
vid = 'videos/sidewalk_walking_1920x1080.mp4'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# initalize deepSORT for tracklet generation

tracker = DeepSORT_Tracker()
track_history = {}

d:\OneDrive\SKS_OneDrive\Projects\Micro-Projects\Computer_Vision\Visitor_Log_Generation\.venv\Lib\site-packages\deep_sort_realtime\embedder\embedder_pytorch.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [6]:
# video inference

def generate_tracklets(vid):
    
    frame_id = 0
    skip = 3 # process 1 in 3 frames
    cap = capture_setter(vid)

    while cap.isOpened():
        ret, frame = cap.read()
        if frame is None or cv2.waitKey(1) & 0xFF == 27: 
            break
        else:
            if frame_id % skip != 0:
                frame_id += 1
                continue

            # person detection
            detections = get_person_detections(frame)

            # person tracking
            tracks_current = tracker.object_tracker.update_tracks(detections, frame = frame)
            tracker.display_track(track_history, tracks_current, frame)

            # tracklet generation
            for track in tracks_current:
                save_tracklets(frame, track)

            # resize frame before display
            resized_frame = cv2.resize(frame, None, fx=0.75, fy=0.75, interpolation=cv2.INTER_AREA)
            cv2.imshow('frame', resized_frame)

    cap.release()
    cv2.destroyAllWindows()

In [7]:
generate_tracklets(vid)

In [8]:
generate_log_from_tracklets(tracklets_dir = 'tracklets/')